# Data Fusion: IoT Telemetry and Contextual Data

This notebook implements a contextual data fusion pipeline for predictive maintenance. We merge internal IoT sensor telemetry with external factory context data by timestamp in order to enrich the feature set and create a more robust dataset for modeling.

## Why contextual data fusion matters

Contextual data fusion combines operational telemetry with external environmental and process context. In predictive maintenance, external context like temperature, humidity, and factory load can provide additional signals that improve failure prediction and help avoid false alerts.

In [ ]:
import pandas as pd
from pathlib import Path

project_root = Path('.')
processed_dir = project_root / 'data' / 'processed'
external_dir = project_root / 'data' / 'external'

ai4i_path = processed_dir / 'ai4i2020_features.csv'
context_path = external_dir / 'context_data.csv'

ai4i_df = pd.read_csv(ai4i_path)
context_df = pd.read_csv(context_path)

print('Loaded AI4I dataset shape:', ai4i_df.shape)
print('Loaded contextual dataset shape:', context_df.shape)

## Generate synthetic timestamps for AI4I telemetry

The AI4I dataset does not include timestamps. For a time-series fusion pipeline, we generate a synthetic timestamp index starting at `2024-01-01 00:00:00` with a `15 minute` frequency. This gives each row a realistic observation time for merge-based alignment.

In [ ]:
start_time = pd.Timestamp('2024-01-01 00:00:00')
frequency = '15T'

a4i_timestamps = pd.date_range(start=start_time, periods=len(ai4i_df), freq=frequency)
ai4i_df['timestamp'] = a4i_timestamps

print('Synthetic timestamp range: {} to {}'.format(ai4i_df['timestamp'].iloc[0], ai4i_df['timestamp'].iloc[-1]))
ai4i_df.head(3)

## Convert timestamps to datetime

We convert both datasets to pandas datetime format so that the merge operation is reliable and time-aware.

In [ ]:
ai4i_df['timestamp'] = pd.to_datetime(ai4i_df['timestamp'])
context_df['timestamp'] = pd.to_datetime(context_df['timestamp'])

print('AI4I timestamp dtype:', ai4i_df['timestamp'].dtype)
print('Context timestamp dtype:', context_df['timestamp'].dtype)

## Align and merge datasets using `merge_asof()`

`merge_asof()` performs a nearest-key merge on sorted time series data. It is useful for realistic alignment of telemetry with contextual readings when timestamps may not match exactly. In this case, both datasets have aligned timestamps, so the merge will join rows by their corresponding observation times.

In [ ]:
ai4i_sorted = ai4i_df.sort_values('timestamp').reset_index(drop=True)
context_sorted = context_df.sort_values('timestamp').reset_index(drop=True)

fused_df = pd.merge_asof(
    ai4i_sorted,
    context_sorted,
    on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('15min')
)

print('Fused dataset shape:', fused_df.shape)

## Fused dataset preview

The merged dataset contains the original AI4I telemetry features, the rolling window features, and the external context variables: `ambient_temperature`, `humidity`, and `factory_load`.

In [ ]:
print('AI4I dataset shape before merge:', ai4i_df.shape)
print('Context dataset shape before merge:', context_df.shape)
print('Fused dataset shape after merge:', fused_df.shape)
print('\nFirst 5 rows of the fused dataset:')
display(fused_df.head())

In [ ]:
missing_values = fused_df.isna().sum()
matched_rows = fused_df['ambient_temperature'].notna().sum()

print('Missing values after merge:')
print(missing_values)
print('\nNumber of rows successfully matched:', matched_rows)

## Why external context improves predictive maintenance

External context data helps explain variation in equipment behavior that sensor data alone may not capture. For example, factory temperature and humidity can influence machine wear rates, while factory load can indicate stress levels. Enriching telemetry with these signals can improve model accuracy and help maintenance teams prioritize interventions.

In [ ]:
output_path = processed_dir / 'fused_dataset.csv'
fused_df.to_csv(output_path, index=False)
print('Saved fused dataset to:', output_path)